In [25]:
import json
from copy import deepcopy
from sklearn.metrics import f1_score, precision_score, recall_score

def get_data(path, preprocessor=None):
    with open(path, "r") as f:
        examples=[json.loads(line) for line in f]
    if preprocessor is not None:
        examples = [preprocessor(example) for example in examples]
    return examples

def preprocess_geo_eval(example):
    instances = []
    i = 0
    while i < len(example["bio_tags"]):
        if example["bio_tags"][i] != "O":
            text = example["tokens"][i]
            tag = example["bio_tags"][i][2:]
            i += 1
            while i < len(example["bio_tags"]) and example["bio_tags"][i] == "I" + example["bio_tags"][i][1:]:
                text += example["tokens"][i] + " "
                i += 1
            instances.append({"type": tag, "text":text})
        i += 1
    return instances

def sample_ner_compare(truths, preds):
    preds_copy = deepcopy(preds)
    results = []
    for instance in truths:
        text_match = False
        type_match = False
        for pred in preds_copy:
            if instance['text'] == pred[0]:
                text_match = "strict"
                type_match = instance['type'] == pred[1]
            elif instance['text'] in pred[0] or pred[0] in instance['text']:
                text_match = "relaxed"
                type_match = instance['type'] == pred[1]

            if text_match != False:
                preds_copy.remove(pred)
                results.append((text_match, type_match, instance['type']))
                break
        if text_match == False:
            results.append((text_match, type_match, instance['type']))
    return results

def get_ner_scores(results):
    strict_text_match = [1 if item[0]=="strict" else 0 for item in results]
    relaxed_text_match = [1 if item[0] in ["relaxed","strict"] else 0 for item in results]
    type_match = [1 if item[1]==True else 0 for item in results]

    return {"strict_text":f1_score([1]*len(strict_text_match), strict_text_match),
            "relaxed_text":f1_score([1]*len(strict_text_match), relaxed_text_match),
            "type":f1_score([1]*len(strict_text_match), type_match)}


In [27]:
test_data = get_data("C:\\Users\\hazza\\OneDrive\\Desktop\\GeoTKG\\cleandata\\geo\\eval.json", preprocessor=preprocess_geo_eval)
geo_ner_preds = get_data("C:\\Users\\hazza\\OneDrive\\Desktop\\GeoTKG\\experiments\\llama3-8B-geoner-preds.json")

In [30]:
results = []
for pred, truth in zip(geo_ner_preds, test_data):
    results.extend(sample_ner_compare(truth, pred['pred']))
get_ner_scores(results)

{'strict_text': 0.5581843191196699,
 'relaxed_text': 0.6879459256477657,
 'type': 0.5760086944708599}